# Day 19 — Does Traffic Explain the B38's Weak Service?

---

## The hypothesis

Your instinct: the B38 runs less frequently in stretches of Bushwick that are farther from subway access, because sparse subway coverage pushes more people into cars, which clogs the same streets the bus has to drive on. Today you build the data to actually test that — three real, independent measurements per B38 stop instead of one dataset doing double duty.

## Two different things people call "bad bus service"

- **Headway** — the *scheduled* gap between buses. This is a decision the MTA makes based on ridership and budget, not something traffic directly changes.
- **Speed / travel time** — how fast the bus actually covers ground. This is exactly what car traffic degrades: a bus stuck behind cars covers less distance per minute, whether or not its schedule changed.

You'll build both today, but keep them conceptually separate — they answer slightly different questions, and conflating them is a common mistake.

## New datasets

| Source | What it gives you | Format note |
|---|---|---|
| MTA Bus Route Segment Speeds | Real measured B38 speed/travel time between timepoints, monthly | Public Socrata API, no key |
| NYC DOT Automated Traffic Volume Counts | Real car counts on B38 corridor streets (DeKalb, Wilson, Knickerbocker, Myrtle Aves) | Public Socrata API, no key — but geometry is **NY State Plane feet, not lat/lon** |
| MTA Subway Stations | All 496 subway station locations | Public Socrata API, no key |
| `data/b38_stops.csv` | Your 10 B38 stops along DeKalb Ave, with a `daily_trips` column | Already on disk |

## New technique: nearest-neighbor spatial join

Day 18's spatial join asked *"does this point fall inside this polygon?"* (`predicate='within'`). Today's question is different: *"which point is closest to this point?"* — there's no containment, just distance. That's `gpd.sjoin_nearest()`, and it comes with its own CRS gotcha you haven't hit yet (more below, when you get there).

## Honest limits before you start

You have **10 B38 stops**. This is an exploratory look, not a statistically powered study — treat any pattern you see as a hypothesis worth investigating further, not a proven result. The traffic sensor data is also sparse by nature (fixed sensor locations, not full street coverage), so absence of a nearby count doesn't mean absence of traffic.

## How today is structured

- **Boilerplate** (API pulls, final plots) — already written.
- **Data work** (aggregating, CRS handling, nearest-neighbor joins, the final merge) — your job.


In [ ]:
# CELL 1 — Imports. (Boilerplate.)
import pandas as pd
import geopandas as gpd
import requests
from shapely import wkt
from shapely.geometry import Point
import matplotlib.pyplot as plt

In [ ]:
# CELL 2 — Pull real B38 bus speed data from the MTA API. (Boilerplate.)
#
# MTA Bus Route Segment Speeds: average measured speed/travel time between
# timepoints (major stops), by route, month, day of week, hour. Public
# Socrata endpoint, no key needed. ~62,000 rows exist for B38 alone across
# all months/hours/directions — we pull all of them and aggregate next.

url = 'https://data.ny.gov/resource/kufs-yh3x.json'
params = {'route_id': 'B38', '$limit': 50000}

response = requests.get(url, params=params)
df_speed = pd.DataFrame(response.json())

# Socrata returns every field as a string, same gotcha as the Census API.
for col in ['average_road_speed', 'average_travel_time', 'timepoint_stop_latitude', 'timepoint_stop_longitude']:
    df_speed[col] = pd.to_numeric(df_speed[col], errors='coerce')

print(df_speed.shape)
print(df_speed[['timepoint_stop_name', 'direction', 'hour_of_day', 'average_road_speed']].head())

**Before Cell 3:** `df_speed` has one row per (timepoint, direction, month, day-of-week, hour) combination — that's why 62,000 rows exist for just 10ish timepoints. To get one meaningful "how fast does the bus usually go here" number per location, what do you need to do to all those rows sharing the same `timepoint_stop_id`? (Same shape of question as Day 18's block-group aggregation — just a new column to collapse by.)

Also worth noticing before you write code: this aggregation throws away *when* — rush hour vs. 3am get averaged together. That's a real simplification, not a bug; keep it in mind when you interpret results later.


In [ ]:
# CELL 3 — Aggregate bus speed to one row per timepoint.
#
# Steps:
#   1. Group df_speed by 'timepoint_stop_id'.
#   2. Aggregate: mean of average_road_speed, and keep one
#      timepoint_stop_name/latitude/longitude per group (they're the same
#      within a group, so 'first' works).
#      Hint: .agg({'average_road_speed': 'mean', 'timepoint_stop_name': 'first',
#                   'timepoint_stop_latitude': 'first', 'timepoint_stop_longitude': 'first'})
#   3. reset_index() — same reason as every day since Day 13: keep the
#      group key as a real column, not the index.
#   4. Store as df_speed_agg.
#
# Sanity check: print(df_speed_agg.shape) — should be roughly a dozen rows,
# not 62,000.

# TODO


In [ ]:
# CELL 4 — Pull real car traffic counts on the B38 corridor. (Boilerplate.)
#
# NYC DOT Automated Traffic Volume Counts, filtered to Brooklyn and to the
# streets the B38 actually runs on. Public Socrata endpoint, no key needed.

corridor_streets = ['DEKALB AVENUE', 'WILSON AVENUE', 'KNICKERBOCKER AVENUE', 'MYRTLE AVENUE']
street_list = "('" + "','".join(corridor_streets) + "')"

url = 'https://data.cityofnewyork.us/resource/7ym2-wayt.json'
params = {
    '$where': f"boro='Brooklyn' AND street in{street_list}",
    '$limit': 50000
}

response = requests.get(url, params=params)
df_traffic_raw = pd.DataFrame(response.json())
df_traffic_raw['vol'] = pd.to_numeric(df_traffic_raw['vol'], errors='coerce')

print(df_traffic_raw.shape)
print(df_traffic_raw[['street', 'fromst', 'tost', 'vol', 'wktgeom']].head(3))

**Before Cell 5 — a new gotcha:** look at the `wktgeom` values you just printed — something like `POINT (1006598.77 195825.25)`. Those are way too large to be latitude/longitude degrees (which top out around ±180). This dataset uses **NY State Plane (EPSG:2263)** — a coordinate system measured in **feet** from a fixed reference point, common in city/state engineering data. It's a real point in Brooklyn, just not in a CRS you can plot next to your other lat/lon data yet.

Two questions before you code:
1. What's the general fix, in one sentence, given what you learned about CRS in Day 18?
2. `wktgeom` is a plain text string, not a geometry object yet — like PLUTO's `latitude`/`longitude` being two separate float columns before you built `Point`s. What shapely function turns WKT text directly into a geometry object? (Hint: it's imported at the top of this notebook already.)


In [ ]:
# CELL 5 — Fix the CRS and aggregate traffic volume by segment.
#
# Steps:
#   1. Parse the WKT strings into geometry objects:
#        geometry = df_traffic_raw['wktgeom'].apply(wkt.loads)
#   2. Wrap into a GeoDataFrame in the CORRECT starting CRS — this data is
#      NY State Plane, so: gpd.GeoDataFrame(df_traffic_raw, geometry=geometry, crs='EPSG:2263')
#      Gotcha: setting crs='EPSG:2263' does NOT convert anything — it just
#      labels the coordinates you already have as being in that system.
#      Setting the wrong CRS label here would silently corrupt every join
#      you do later, with no error.
#   3. Reproject to EPSG:4326 to match everything else you're building today:
#      .to_crs('EPSG:4326')
#   4. Group by 'segmentid', take the mean of 'vol', keep 'street' and
#      geometry via 'first', reset_index(). Store as gdf_traffic_agg.
#
# Sanity check: print(gdf_traffic_agg.crs) and print(gdf_traffic_agg[['street','vol']].head())

# TODO


In [ ]:
# CELL 6 — Pull subway station locations, and load your B38 stops. (Boilerplate.)

url = 'https://data.ny.gov/resource/39hk-dx4f.json'
response = requests.get(url, params={'$limit': 1000})
df_subway = pd.DataFrame(response.json())
df_subway['gtfs_latitude'] = pd.to_numeric(df_subway['gtfs_latitude'], errors='coerce')
df_subway['gtfs_longitude'] = pd.to_numeric(df_subway['gtfs_longitude'], errors='coerce')

df_b38 = pd.read_csv('../data/b38_stops.csv')

print(df_subway.shape, df_b38.shape)

**Before Cell 7:** B38 buses run roughly 5am to 11pm — call it an 18-hour service span, 1,080 minutes. If a stop shows `daily_trips = 85`, what's the approximate average headway in minutes? Write the ratio out before you code it — this is the same rate logic as any other "total ÷ count" average.

Also: you've built a GeoDataFrame from lat/lon columns twice already this project (Day 18, and two cells ago). Without looking back — what are the three ingredients?


In [ ]:
# CELL 7 — Compute approximate headway, and build the two remaining GeoDataFrames.
#
# Steps:
#   1. df_b38['headway_min'] = 1080 / df_b38['daily_trips']
#   2. Build gdf_b38: Point geometry from df_b38's longitude/latitude,
#      crs='EPSG:4326'.
#   3. Build gdf_subway: Point geometry from df_subway's gtfs_longitude/
#      gtfs_latitude, crs='EPSG:4326'.
#
# Sanity check: print(df_b38[['stop_name','daily_trips','headway_min']])

# TODO


**Before Cell 8 — the nearest-neighbor gotcha:** `gpd.sjoin_nearest()` measures distance in whatever units your CRS uses. Both `gdf_b38` and `gdf_subway` are currently in EPSG:4326 — plain lat/lon *degrees*. If you ran the join right now, the distance column would come back in **degrees of latitude/longitude**, a unit that doesn't correspond to a fixed real-world distance and is genuinely meaningless to a reader (1 degree of longitude is a very different real distance in Brooklyn than at the equator).

The fix follows directly from Cell 5: reproject to a CRS measured in real units — EPSG:2263 (NY State Plane, **feet**) — right before the distance calculation, then convert feet to miles after.

Worked pattern for the subway distance (study this, then repeat the shape of it twice more in the next cell for traffic and speed):

```python
gdf_b38_ft = gdf_b38.to_crs('EPSG:2263')
gdf_subway_ft = gdf_subway.to_crs('EPSG:2263')

joined = gpd.sjoin_nearest(gdf_b38_ft, gdf_subway_ft, distance_col='dist_ft')
gdf_b38['dist_to_subway_mi'] = joined['dist_ft'].values / 5280
```


In [ ]:
# CELL 8 — Nearest-neighbor join: attach subway distance, nearest traffic
# volume, and nearest bus speed to every B38 stop.
#
# Part 1 (subway) is done for you above — paste it in below to start.
#
# Then repeat the same 3-line shape twice more:
#   - gdf_traffic_agg (from Cell 5) -> bring over 'vol' as 'nearest_traffic_vol'
#   - df_speed_agg (from Cell 3) — build a quick GeoDataFrame from its lat/lon
#     first, same pattern as Cell 7 -> bring over 'average_road_speed' as
#     'nearest_avg_speed'
#
# Gotcha carried over from every merge you've done since Day 4: check
# len(gdf_b38) before and after each join — sjoin_nearest can duplicate a
# row if two candidates are exactly tied for closest.
#
# Sanity check: print(gdf_b38[['stop_name','headway_min','dist_to_subway_mi',
#                               'nearest_traffic_vol','nearest_avg_speed']])

# TODO


In [ ]:
# CELL 9 — Does the pattern show up? (Boilerplate.)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(gdf_b38['dist_to_subway_mi'], gdf_b38['nearest_avg_speed'])
axes[0].set_xlabel('Distance to nearest subway (mi)')
axes[0].set_ylabel('Nearest measured bus speed (mph)')
axes[0].set_title('Subway distance vs. bus speed')

axes[1].scatter(gdf_b38['nearest_traffic_vol'], gdf_b38['nearest_avg_speed'])
axes[1].set_xlabel('Nearest car traffic volume')
axes[1].set_ylabel('Nearest measured bus speed (mph)')
axes[1].set_title('Traffic volume vs. bus speed')

axes[2].scatter(gdf_b38['dist_to_subway_mi'], gdf_b38['headway_min'])
axes[2].set_xlabel('Distance to nearest subway (mi)')
axes[2].set_ylabel('Approx. headway (min)')
axes[2].set_title('Subway distance vs. headway')

plt.tight_layout()
plt.show()

**Critical thinking pause:**
- Which of the three panels shows the clearest pattern, if any? Does it point the direction your hypothesis predicted?
- With only 10 stops, how much should one or two outlier stops be allowed to shape your conclusion?
- Name one confound you haven't controlled for — something else that changes along DeKalb Ave from Broadway to Pennsylvania Ave besides subway distance and traffic (population density, street width, zoning, number of traffic lights...). Could it explain the pattern instead of, or in addition to, traffic?
- `nearest_traffic_vol` comes from sparse, fixed sensor locations, not full street coverage. If a stop's nearest sensor is a mile away, how much should you trust that number as "the traffic at this stop"?
- You built `headway_min` from a flat 1,080-minute assumption for every stop. What would make that assumption wrong in a way that biases the headway column specifically (not just adds random noise)?


*(Write your answers here)*

## Day 19 Recap

| What you did | Why it matters |
|---|---|
| Distinguished headway from speed | Two different metrics, two different causes — conflating them is a common real-world analysis mistake |
| Pulled 3 independent real datasets | Avoided circular reasoning — traffic, subway distance, and bus performance are each measured separately, not inferred from each other |
| Parsed WKT + fixed a State Plane CRS | Real government data doesn't always hand you lat/lon — recognizing an unfamiliar CRS from the numbers themselves is a transferable skill |
| `gpd.sjoin_nearest()` | A second spatial join type — distance-based, not containment-based |
| Reprojected before measuring distance | Distance in degrees is meaningless; distance in feet/miles is a real number a reader can use |

## Close-out

In your own words: why do you need to reproject to a CRS like EPSG:2263 before computing a distance with `sjoin_nearest`, even though your data already "has" a CRS (EPSG:4326)? What would the distance column mean if you skipped that step?

### What's coming on Day 20
TBD — options on the table: ACRIS ownership chains (who owns multiple Bushwick lots, and does that correlate with risk score), or folding this traffic/subway analysis back into the walk-07-far site as a third map layer.
